# LaminoRAG Demo

Personal laminopathy research assistant. Forced cross-corpus retrieval over the user's reading.

## Setup

Add repo root to `sys.path` so notebook can import sibling modules. Verify config + env vars.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import config
print("Corpora:", config.ALL_CORPORA)
print("Embedding backend:", config.EMBEDDING_BACKEND)
print("Chroma path:", config.CHROMA_PATH)
print("LLM model:", config.LLM_MODEL)
print("LLM_API_BASE set:", bool(os.environ.get("LLM_API_BASE")))
print("LLM_API_KEY set:", bool(os.environ.get("LLM_API_KEY")))
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("neuml/pubmedbert-base-embeddings")
print("Tokenizer loaded:", tok)

Corpora: ['laminopathy', 'lnp', 'bioinformatics']
Embedding backend: pubmedbert
Chroma path: c:\Users\Vladislav Karpe\Desktop\VSE\6semestr\sythesis\data\chroma_db
LLM model: qwen3.6-35b
LLM_API_BASE set: True
LLM_API_KEY set: True


## Ingest

Drop articles into `data/articles/<corpus>/` (e.g. `data/articles/laminopathy/foo.pdf`). The parent folder name determines the corpus. Re-ingesting the same file replaces its prior chunks.

In [ ]:
import ingest

articles_dir = ROOT / "data" / "articles"
for article in sorted(articles_dir.rglob("*")):
    if article.suffix.lower() not in {".pdf", ".txt", ".md"}:
        continue
    result = ingest.ingest(article)
    print(f"{article.relative_to(articles_dir)}: {result}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Query — laminopathy-centered

In [ ]:
import query

response = query.ask(
    "Read the connections between changed gene expressions in laminopathic hearts "
    "and propose an experiment to find commonality between them."
)
print("ANSWER:\n")
print(response["answer"])
print("\nSOURCES BY CORPUS:")
for corpus, titles in response["sources"].items():
    print(f"  {corpus}: {titles}")

## Query — cross-corpus

In [ ]:
response = query.ask("Can bioinformatics approaches improve LNP targeting for laminopathy?")
print("ANSWER:\n")
print(response["answer"])
print("\nSOURCES BY CORPUS:")
for corpus, titles in response["sources"].items():
    print(f"  {corpus}: {titles}")